# Qiskit Fall Fest — Applied Practice Lab
### From Fundamentals to Circuit Reasoning

This lab assumes you have completed the beginner teaching notebook.

It deliberately **does not repeat the lecture challenges** such as simply preparing $|1\rangle$, $|+\rangle$, a Bell state, or a GHZ state from a recipe.

Instead, you will apply the same ideas in less familiar situations.

## Lab rule

For each problem:

1. **Predict** before running.
2. **Build or repair** the circuit.
3. **Run** it.
4. **Explain** the result.
5. **Change one thing** and decide whether your explanation still works.

Displayed equations use `$$ ... $$` for reliable notebook rendering.


## 0. Setup

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram, plot_bloch_multivector
import numpy as np

simulator = AerSimulator()
print("Lab ready.")


# 1. Reverse Engineering a Circuit

You are given a circuit rather than a target state.

**Do not run it immediately.**

```text
q0: H → Z → H
```

Answer first:

- What state does the qubit begin in?
- What do you think the final computational-basis state will be?
- Will measurement be deterministic or probabilistic?

Then use Qiskit to test your reasoning.


In [ ]:
qc = QuantumCircuit(1)

qc.h(0)
qc.z(0)
qc.h(0)

# Inspect before measurement
state = Statevector.from_instruction(qc)
print(state)
plot_bloch_multivector(state)

qc.draw("mpl")


In [ ]:
# Now measure the same transformation

measured = QuantumCircuit(1, 1)
measured.h(0)
measured.z(0)
measured.h(0)
measured.measure(0, 0)

result = simulator.run(measured, shots=1000).result()
print(result.get_counts())


### Extension

Find another sequence of gates that produces the **same measured result**.

Then determine whether the two circuits also have the same statevector before measurement.


In [ ]:
# Your alternative circuit

qc2 = QuantumCircuit(1)

# Add gates


print(Statevector.from_instruction(qc2))
qc2.draw("mpl")


# 2. Same Counts, Different States

A computational-basis measurement does not reveal everything about a quantum state.

Construct these two states:

$$
|+\rangle = \frac{|0\rangle+|1\rangle}{\sqrt{2}}
$$

and

$$
|-\rangle = \frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

You already learned how to prepare them in class. The challenge here is different:

> Can ordinary computational-basis measurement distinguish them?

Predict first, then test both circuits with 4000 shots.


In [ ]:
plus = QuantumCircuit(1, 1)
plus.h(0)
plus.measure(0, 0)

minus = QuantumCircuit(1, 1)
minus.x(0)
minus.h(0)
minus.measure(0, 0)

plus_counts = simulator.run(plus, shots=4000).result().get_counts()
minus_counts = simulator.run(minus, shots=4000).result().get_counts()

print("|+> counts:", plus_counts)
print("|-> counts:", minus_counts)


### Challenge: change the measurement basis

Before measurement, add a gate that allows the difference between $|+\rangle$ and $|-\rangle$ to become visible in computational-basis measurements.

**Hint:** Which gate connects the computational basis and the $+$/$-$ basis?


In [ ]:
# Modify these circuits so the difference becomes visible.

plus_test = QuantumCircuit(1, 1)
plus_test.h(0)

# Add basis-change gate here


plus_test.measure(0, 0)

minus_test = QuantumCircuit(1, 1)
minus_test.x(0)
minus_test.h(0)

# Add basis-change gate here


minus_test.measure(0, 0)

print("Plus:", simulator.run(plus_test, shots=1000).result().get_counts())
print("Minus:", simulator.run(minus_test, shots=1000).result().get_counts())


# 3. Design a Probability

Instead of being told which gate to use, you are given a target behaviour.

Build a one-qubit circuit whose measurement returns approximately:

- `0` with probability 75%
- `1` with probability 25%

Use an $R_y(\theta)$ rotation.

For

$$
R_y(\theta)|0\rangle,
$$

the probability of measuring `1` is

$$
P(1)=\sin^2\left(\frac{\theta}{2}\right).
$$

Find a suitable value of $\theta$, then test it with 10,000 shots.


In [ ]:
# Choose theta from the target probability.

theta = 0  # replace this

qc = QuantumCircuit(1, 1)
qc.ry(theta, 0)
qc.measure(0, 0)

result = simulator.run(qc, shots=10000).result()
counts = result.get_counts()

print("theta =", theta)
print(counts)
plot_histogram(counts)


### Extension

Change the target to:

$$
P(1)=0.10
$$

Can you determine an angle rather than finding it by random trial and error?


# 4. Measurement Mapping Puzzle

This exercise tests whether you understand **where measurement information is stored**.

Consider three qubits and three classical bits.

Prepare:

```text
q0 = 1
q1 = 0
q2 = 1
```

but store the measurements using:

```python
qc.measure([0, 1, 2], [2, 0, 1])
```

Before running the circuit, predict the bit string Qiskit will display.

Remember that Qiskit displays the higher-index classical bit on the left.


In [ ]:
qc = QuantumCircuit(3, 3)

qc.x(0)
qc.x(2)

qc.measure([0, 1, 2], [2, 0, 1])

qc.draw("mpl")

result = simulator.run(qc, shots=100).result()
print(result.get_counts())


### Mapping challenge

Change only the **measurement mapping**, not the quantum state, so that the displayed result becomes `101`.

Then create another mapping that displays `110`.


In [ ]:
qc = QuantumCircuit(3, 3)

qc.x(0)
qc.x(2)

# Change only this measurement mapping:
# qc.measure(..., ...)


result = simulator.run(qc, shots=100).result()
print(result.get_counts())


# 5. Diagnose the Broken Entanglement Experiment

A student says:

> "I built an entangled state because I used H and CX."

Their circuit is:


In [ ]:
broken = QuantumCircuit(2, 2)

broken.h(0)
broken.h(1)
broken.cx(0, 1)

broken.measure([0, 1], [0, 1])

broken.draw("mpl")

counts = simulator.run(broken, shots=4000).result().get_counts()
print(counts)
plot_histogram(counts)


Answer:

1. What outcomes do you expect?
2. Does merely including H and CX guarantee the Bell-state behaviour seen in class?
3. Modify the circuit by **removing or changing as little as possible** so that the familiar `00`/`11` Bell correlations appear.
4. Explain why your modification works.


In [ ]:
fixed = QuantumCircuit(2, 2)

# Repair the circuit with the smallest useful change


fixed.measure([0, 1], [0, 1])

counts = simulator.run(fixed, shots=4000).result().get_counts()
print(counts)
plot_histogram(counts)


# 6. Correlation Without Looking at the Circuit

Two unknown circuits were run 4000 times.

One produced roughly:

```text
00: 1000
01: 1000
10: 1000
11: 1000
```

The other produced roughly:

```text
00: 2000
11: 2000
```

For each result:

- What can you say about the relationship between the measured bits?
- Which result is consistent with the Bell-state experiment from class?
- Why is "each qubit is 50/50" not enough to describe the difference?

Now build two circuits that reproduce these two kinds of distributions.


In [ ]:
# Circuit A

qc_a = QuantumCircuit(2, 2)

# Build it


qc_a.measure([0, 1], [0, 1])

# Circuit B

qc_b = QuantumCircuit(2, 2)

# Build it


qc_b.measure([0, 1], [0, 1])

print("A:", simulator.run(qc_a, shots=4000).result().get_counts())
print("B:", simulator.run(qc_b, shots=4000).result().get_counts())


# 7. Bell-State Decoder

Suppose an unknown circuit prepares one of these states:

$$
|\Phi^+\rangle=\frac{|00\rangle+|11\rangle}{\sqrt2}
$$

$$
|\Phi^-\rangle=\frac{|00\rangle-|11\rangle}{\sqrt2}
$$

$$
|\Psi^+\rangle=\frac{|01\rangle+|10\rangle}{\sqrt2}
$$

$$
|\Psi^-\rangle=\frac{|01\rangle-|10\rangle}{\sqrt2}
$$

Computational-basis measurement alone cannot reveal every sign difference.

Your task is to build a **decoder** using the inverse of the Bell-state preparation:

1. apply `cx(0, 1)`;
2. apply `h(0)`;
3. measure.

First test the decoder on the standard Bell state from class. Then use X and/or Z to create Bell-state variations and see whether the decoder produces distinct deterministic outputs.


In [ ]:
# Prepare a Bell-state variation, then decode it.

qc = QuantumCircuit(2, 2)

# --- State preparation ---
qc.h(0)
qc.cx(0, 1)

# Modify with X and/or Z here if desired


qc.barrier()

# --- Bell decoder ---
qc.cx(0, 1)
qc.h(0)

qc.measure([0, 1], [0, 1])

qc.draw("mpl")

result = simulator.run(qc, shots=1000).result()
print(result.get_counts())


# 8. Build a SWAP Without `swap()`

You saw the SWAP gate in class.

Now reproduce its behaviour **without using `qc.swap()`**.

A SWAP can be constructed from three CNOT operations.

Your job is to determine the control/target ordering.

Test your circuit on at least these inputs:

- `01`
- `10`
- `11`

The first two should exchange; `11` should remain `11`.


In [ ]:
qc = QuantumCircuit(2, 2)

# Prepare a test input.
qc.x(0)

# Build SWAP using exactly three CX gates.
# qc.cx(...)
# qc.cx(...)
# qc.cx(...)

qc.measure([0, 1], [0, 1])

qc.draw("mpl")

print(simulator.run(qc, shots=1000).result().get_counts())


# 9. GHZ Under Disturbance

In class you built:

$$
|GHZ\rangle =
\frac{|000\rangle+|111\rangle}{\sqrt2}.
$$

Here the challenge is not to build it again. Instead, investigate what happens when it is disturbed.

Start with a GHZ circuit and insert **one X gate on qubit 1 after entanglement**.

Before running, predict which two bit strings will dominate.

Then repeat with:

- X on qubit 0;
- X on qubit 2;
- Z on one qubit.

Which disturbances change computational-basis counts? Which may require another measurement basis to detect?


In [ ]:
qc = QuantumCircuit(3, 3)

# GHZ preparation
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)

# Disturb the state here
qc.x(1)

qc.measure([0, 1, 2], [0, 1, 2])

result = simulator.run(qc, shots=4000).result()
counts = result.get_counts()

print(counts)
plot_histogram(counts)


# 10. Partial Measurement

Create a three-qubit GHZ state, but measure **only one qubit** into one classical bit.

Compare this with measuring all three qubits.

Questions:

- What information is lost when only one qubit is measured?
- Can the one-bit counts reveal the three-qubit correlation?
- Why can two experiments both show a 50/50 single-bit result while representing very different multi-qubit states?


In [ ]:
qc = QuantumCircuit(3, 1)

qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)

# Measure only one qubit
qc.measure(0, 0)

result = simulator.run(qc, shots=4000).result()
print(result.get_counts())

qc.draw("mpl")


# 11. Circuit Equivalence Challenge

Compare these two circuits:

### Circuit A

```text
H → H
```

### Circuit B

```text
(no gates)
```

They should behave the same on $|0\rangle$.

But does that mean they are always interchangeable?

Test them on both $|0\rangle$ and $|1\rangle$.

Then investigate another pair:

```text
X → X
```

versus no gates.

Use both statevectors and measurement results.


In [ ]:
# Build an input state, then compare two transformations.

input_is_one = False

a = QuantumCircuit(1)
b = QuantumCircuit(1)

if input_is_one:
    a.x(0)
    b.x(0)

# Transformation A
a.h(0)
a.h(0)

# Transformation B: intentionally empty

print("A:", Statevector.from_instruction(a))
print("B:", Statevector.from_instruction(b))


# 12. Transpilation Investigation

The lecture introduced transpilation. Here you will inspect it rather than merely call it.

Build a circuit containing:

- H
- CX
- SWAP

Then transpile it using a restricted basis:

```python
basis_gates=["rz", "sx", "x", "cx"]
```

Compare:

- `count_ops()`
- `depth()`
- the original circuit
- the transpiled circuit

What happened to SWAP?


In [ ]:
qc = QuantumCircuit(3)

qc.h(0)
qc.cx(0, 1)
qc.swap(1, 2)

compiled = transpile(
    qc,
    basis_gates=["rz", "sx", "x", "cx"],
    optimization_level=1
)

print("ORIGINAL")
print(qc)
print("Operations:", qc.count_ops())
print("Depth:", qc.depth())

print("\nTRANSPILED")
print(compiled)
print("Operations:", compiled.count_ops())
print("Depth:", compiled.depth())


# 13. Optimization Challenge

Create a deliberately redundant circuit containing operations such as repeated X or H gates.

Transpile it using different optimization levels:

```python
optimization_level=0
optimization_level=1
optimization_level=3
```

Compare circuit depth and operation counts.

Questions:

- Which redundant operations disappear?
- Does a shorter circuit necessarily mean a different computation?
- Why might reducing unnecessary operations matter on real hardware?


In [ ]:
qc = QuantumCircuit(2)

# Deliberately redundant circuit
qc.h(0)
qc.h(0)
qc.x(1)
qc.x(1)
qc.h(0)
qc.cx(0, 1)

for level in [0, 1, 3]:
    compiled = transpile(qc, simulator, optimization_level=level)
    print(f"\nOptimization level {level}")
    print("Depth:", compiled.depth())
    print("Operations:", compiled.count_ops())
    print(compiled)


# 14. Mystery Circuit Challenge

The circuit below is intentionally hidden behind a function.

Run it for different inputs and determine what transformation it performs.

Do not inspect the function body until you have formed a hypothesis.


In [ ]:
def mystery_circuit(input_bit):
    qc = QuantumCircuit(1, 1)

    if input_bit == 1:
        qc.x(0)

    qc.h(0)
    qc.z(0)
    qc.h(0)

    qc.measure(0, 0)
    return qc

for input_bit in [0, 1]:
    qc = mystery_circuit(input_bit)
    counts = simulator.run(qc, shots=1000).result().get_counts()
    print("Input:", input_bit, "Output:", counts)


### Your task

From the input/output behaviour:

1. What familiar gate does the middle sequence appear to reproduce?
2. Build a simpler circuit that performs the same transformation.
3. Test both on the same inputs.

Only after making your prediction should you reason back through the sequence.


In [ ]:
# Simpler equivalent circuit

qc = QuantumCircuit(1, 1)

# Your replacement operation


qc.measure(0, 0)


# 15. Final Applied Challenge — Design an Experiment

Choose **one property from class** and investigate it beyond the original example.

Your experiment must use at least two of the following:

- statevector inspection;
- a change of measurement basis;
- custom measurement mapping;
- multi-qubit correlations;
- gate equivalence;
- controlled operations;
- circuit transpilation.

### Possible research questions

- How can phase differences be made visible through measurement?
- Can two different circuits produce the same counts but different states?
- How does disturbing one qubit affect GHZ correlations?
- How can a familiar gate be decomposed into simpler gates?
- How does measurement mapping change the bit string without changing the quantum state?
- How does transpilation change a circuit while preserving its intended computation?

### Submission

Include:

1. your question;
2. your prediction;
3. your circuit;
4. your result;
5. your explanation;
6. one modification;
7. a second prediction and result;
8. what you learned from the comparison.


In [ ]:
# FINAL APPLIED CHALLENGE

# Build your experiment here.



# Completion Check

By the end of this lab, you should be doing more than reproducing examples.

You should be able to:

- reason backwards from a circuit or result;
- distinguish state information from measurement information;
- deliberately change measurement basis;
- control where measurement data is stored;
- reason about correlations rather than only individual probabilities;
- modify Bell and GHZ experiments;
- decompose a higher-level operation;
- compare equivalent circuits;
- inspect the effect of transpilation;
- design a small quantum experiment from a question rather than a recipe.

Those skills are the bridge from the introductory notebook to more advanced Qiskit work.
